# PCA on MSU sites
This notebook performs a windowed read of images over MSU sites, extracts embeddings and performs PCA.

1. Perform the windowed read to split imagery into 512 x 512 chips in order to feed into inference code. 
2. Perform the image preprocessing.
3. Extract the embeddings.
4. Perform PCA.

MSU test site is cerath or ileg.

In [ ]:
import pickle
import os
import urllib
import torch

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torchvision.transforms.functional as TF
from sklearn.decomposition import PCA
from scipy import signal

import sys
import os
from pathlib import Path

import rasterio
from rasterio.plot import show
from rasterio.windows import Window

sys.path.insert(0, str(Path("../dinov3").resolve()))
sys.path.insert(0, str(Path('../src').resolve()))

import run_pca

%load_ext autoreload
%autoreload 2

In [ ]:
import torch
print(">>> TORCH VERSION", torch.__version__)
print(torch.__file__)

import sys
print(sys.executable)

## Run PCA inference pipeline 

### test site: KEV

In [ ]:
run_pca.infer_overlap_blend_possub(
        "../data/kev_2023-07-21_cropped.tif",
        "../data/kev_2023-07-21_pca_overlap_blend_2.tif",
        pos_template_max_windows=200,
        pca_sample_per_window=256,
        scale_lo_pct=2.0,
        scale_hi_pct=98.0,
    )

In [ ]:
# test run infer pipe
# used smaller spatial win and img size, dropped whitening, reduced target sample, switched to dinov3_vitb16
debug, out = run_pca.infer_pipe("../data/kev_2023-07-21_cropped.tif",
                   "../data/kev_2023-07-21_pca_vitl16_weights.tif")

In [ ]:
import seaborn as sns
sns.heatmap(out['samples'][0].feats[55])

In [ ]:
def show_debug_sample(sample):
    import matplotlib.pyplot as plt
    fig, axs = plt.subplots(1, 3, figsize=(12, 4))
    axs[0].imshow(sample.input_rgb_u8); axs[0].set_title("Input (cropped)"); axs[0].axis("off")
    axs[1].imshow(sample.vis_tok_u8); axs[1].set_title("PC viz (token grid)"); axs[1].axis("off")
    axs[2].imshow(sample.vis_resized_u8); axs[2].set_title("PC viz resized to input"); axs[2].axis("off")
    #axs[2].imshow(sample.feats); axs[2].set_title("PC viz resized to input"); axs[2].axis("off")
    plt.tight_layout()
    plt.show()

show_debug_sample(out['samples'][1])

In [ ]:
import seaborn as sns
sns.heatmap(out['samples'][0]

### test site: Cerath

In [ ]:
run_pca.infer_pipe('../data/msu_images/cerath_2023-10-01_cropped.tif')

# John's Example Code

In [ ]:
from dinov3.hub.backbones import dinov3_vit7b16, dinov3_vitl16
from PIL import Image
import urllib

model = dinov3_vitl16(pretrained = False)

In [ ]:
model.rope_embed

In [ ]:
from dinov3.hub.backbones import dinov3_vit7b16, dinov3_vitl16
from PIL import Image
import urllib

model = dinov3_vitl16(pretrained = False)
url = "../dinov3/dinov3/weights/dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth"
state_dict = torch.load(url, map_location="cpu")
model.load_state_dict(state_dict, strict=False)

PATCH_SIZE = 16
IMAGE_SIZE = 1536

MEAN = (0.430, 0.411, 0.296 )
STD = (0.213, 0.156, 0.143 )

def resize_transform(
    mask_image: Image,
    image_size: int = IMAGE_SIZE,
    patch_size: int = PATCH_SIZE,
) -> torch.Tensor:
    w, h = mask_image.size
    h_patches = int(image_size / patch_size)
    w_patches = int((w * image_size) / (h * patch_size))
    return TF.to_tensor(TF.resize(mask_image, (h_patches * patch_size, w_patches * patch_size)))

from PIL import Image
image = Image.open("urban_248.jpg")

image_resized = resize_transform(image)
image_resized_norm = TF.normalize(image_resized, mean=MEAN, std=STD)

n_layers = 24
with torch.inference_mode():
    with torch.no_grad():
        feats = model.get_intermediate_layers(image_resized_norm.unsqueeze(0), n=range(n_layers), reshape=True, norm=True)
        x = feats[-1].squeeze().detach().cpu()
        dim = x.shape[0]
        x = x.view(dim, -1).permute(1, 0)
pca = PCA(n_components=3, whiten=True)
pca.fit(x)


In [ ]:
h_patches = image_resized.shape[1] // 16
w_patches = image_resized.shape[2] // 16
projected_image = torch.from_numpy(pca.transform(x.numpy())).view(h_patches, w_patches, 3)

# multiply by 2.0 and pass through a sigmoid to get vibrant colors 
projected_image = torch.nn.functional.sigmoid(projected_image.mul(2.0)).permute(2, 0, 1)
# enjoy
plt.figure(dpi=200)
plt.imshow(projected_image.permute(1, 2, 0))
plt.axis('off')
plt.show()

In [ ]:
image